# Kilter Board — Exploratory Data Analysis

Explore the Kilter Board database: grade distributions, hold positions, angle effects, and per-hold usage patterns.

In [ ]:
import sys

sys.path.insert(0, "..")

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import seaborn as sns  # noqa: E402

from src.data.ingest import load_climbs, load_difficulty_grades, load_placements  # noqa: E402

sns.set_theme(style="whitegrid", palette="viridis")
pd.set_option("display.max_columns", 20)

DB_PATH = "../data/raw/kilter.db"

In [ ]:
df = load_climbs(DB_PATH, min_ascents=5)
grades_map = load_difficulty_grades(DB_PATH)
placements = load_placements(DB_PATH)

print(f"Routes (route-angle combos): {len(df):,}")
print(f"Unique routes: {df['climb_uuid'].nunique():,}")
print(f"Angles: {sorted(df['angle'].unique())}")
print(f"Grade range: {df['grade'].min():.1f} – {df['grade'].max():.1f}")
print(f"Hold count range: {df['hold_count'].min()} – {df['hold_count'].max()}")
df.head(3)

## 1. Grade Distribution

In [ ]:
# Map numeric grades to V-grade labels for readability
grade_labels = grades_map.set_index("difficulty")["boulder_name"].to_dict()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of continuous grade
axes[0].hist(df["grade"], bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Grade (continuous)")
axes[0].set_ylabel("Count")
axes[0].set_title("Grade Distribution (continuous scale)")

# Bar chart by V-grade bucket
df["grade_rounded"] = df["grade"].round().astype(int)
df["v_grade"] = df["grade_rounded"].map(grade_labels).fillna("Unknown")
grade_order = grades_map.sort_values("difficulty")["boulder_name"].tolist()
grade_counts = df["v_grade"].value_counts().reindex(grade_order).dropna()
axes[1].bar(range(len(grade_counts)), grade_counts.values, edgecolor="black", alpha=0.7)
axes[1].set_xticks(range(len(grade_counts)))
axes[1].set_xticklabels(grade_counts.index, rotation=45, ha="right")
axes[1].set_ylabel("Count")
axes[1].set_title("Routes per V-Grade")

plt.tight_layout()
plt.show()

median_grade = df["grade"].median()
median_label = grade_labels.get(int(round(median_grade)), "?")
print(f"Median grade: {median_grade:.1f} ({median_label})")
print(f"Mean grade: {df['grade'].mean():.1f}")

## 2. Angle Distribution & Grade vs Angle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Angle distribution
angle_counts = df["angle"].value_counts().sort_index()
axes[0].bar(angle_counts.index, angle_counts.values, width=4, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Angle (degrees)")
axes[0].set_ylabel("Count")
axes[0].set_title("Routes per Angle")

# Grade vs angle boxplot
axes[1].boxplot(
    [df[df["angle"] == a]["grade"].values for a in sorted(df["angle"].unique())],
    positions=sorted(df["angle"].unique()),
    widths=3,
    manage_ticks=False,
)
axes[1].set_xlabel("Angle (degrees)")
axes[1].set_ylabel("Grade")
axes[1].set_title("Grade Distribution by Angle")

plt.tight_layout()
plt.show()

# Mean grade per angle
mean_by_angle = df.groupby("angle")["grade"].mean()
print("Mean grade by angle:")
for angle, mean in mean_by_angle.items():
    print(f"  {angle:2d}°: {mean:.1f} ({grade_labels.get(int(round(mean)), '?')})")

## 3. Hold Position Heatmap\n\nWhich positions on the board are most frequently used across all routes?

In [ ]:
# Explode holds to get one row per hold usage
holds_flat = df.explode("holds").reset_index(drop=True)
holds_expanded = pd.json_normalize(holds_flat["holds"])
holds_expanded["grade"] = holds_flat["grade"].values
holds_expanded["angle"] = holds_flat["angle"].values

fig, axes = plt.subplots(1, 2, figsize=(12, 14))

# Usage frequency heatmap
usage = holds_expanded.groupby(["x", "y"]).size().reset_index(name="count")
scatter = axes[0].scatter(
    usage["x"],
    usage["y"],
    c=usage["count"],
    s=usage["count"] / usage["count"].max() * 200,
    cmap="YlOrRd",
    alpha=0.8,
    edgecolors="black",
    linewidths=0.3,
)
axes[0].set_xlabel("X position")
axes[0].set_ylabel("Y position")
axes[0].set_title("Hold Usage Frequency")
plt.colorbar(scatter, ax=axes[0], label="Times used")

# All available hold positions (grey) for reference
axes[1].scatter(
    placements["x"],
    placements["y"],
    c="lightgrey",
    s=15,
    alpha=0.5,
    label="All holds",
)
# Overlay top 50 most used holds
top_holds = usage.nlargest(50, "count")
scatter2 = axes[1].scatter(
    top_holds["x"],
    top_holds["y"],
    c=top_holds["count"],
    s=80,
    cmap="YlOrRd",
    edgecolors="black",
    linewidths=0.5,
    label="Top 50 used",
)
axes[1].set_xlabel("X position")
axes[1].set_ylabel("Y position")
axes[1].set_title("Top 50 Most Used Holds")
axes[1].legend(loc="upper right")
plt.colorbar(scatter2, ax=axes[1], label="Times used")

plt.tight_layout()
plt.show()

## 4. Hold Count vs Grade

In [ ]:
# Filter out outlier hold counts for cleaner visualization
df_viz = df[df["hold_count"] <= 30]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(df_viz["hold_count"], df_viz["grade"], alpha=0.05, s=5)
axes[0].set_xlabel("Hold Count")
axes[0].set_ylabel("Grade")
axes[0].set_title("Hold Count vs Grade")

# Mean grade by hold count
mean_by_holds = df_viz.groupby("hold_count")["grade"].agg(["mean", "std", "count"])
mean_by_holds = mean_by_holds[mean_by_holds["count"] >= 10]
axes[1].errorbar(
    mean_by_holds.index,
    mean_by_holds["mean"],
    yerr=mean_by_holds["std"],
    fmt="o-",
    capsize=3,
    markersize=4,
)
axes[1].set_xlabel("Hold Count")
axes[1].set_ylabel("Mean Grade ± 1 SD")
axes[1].set_title("Mean Grade by Hold Count")

plt.tight_layout()
plt.show()

print(f"Correlation (hold_count vs grade): {df_viz['hold_count'].corr(df_viz['grade']):.3f}")

## 5. Ascent Count Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df["ascensionist_count"], bins=100, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Ascensionist Count")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Ascent Count Distribution")
axes[0].set_xlim(0, 500)

axes[1].hist(np.log10(df["ascensionist_count"]), bins=50, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("log10(Ascensionist Count)")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Ascent Count Distribution (log scale)")

plt.tight_layout()
plt.show()

print(f"Median ascents: {df['ascensionist_count'].median():.0f}")
print(f"Mean ascents: {df['ascensionist_count'].mean():.0f}")
print(f"Max ascents: {df['ascensionist_count'].max()}")

## 6. Per-Hold Usage Analysis (Hold Usability Preview)\n\nFor each hold position, compute the mean grade of routes it appears in. This previews the \"hold usability\" feature we'll engineer in Phase 3 — the hypothesis is that holds appearing in easy routes even at steep angles are \"good\" holds (jug-like), while holds only appearing in hard routes are \"bad\" holds (crimp-like).

In [ ]:
# Per-hold statistics: mean grade, usage count, mean angle
hold_stats = (
    holds_expanded.groupby(["x", "y"])
    .agg(
        mean_grade=("grade", "mean"),
        std_grade=("grade", "std"),
        mean_angle=("angle", "mean"),
        usage_count=("grade", "size"),
    )
    .reset_index()
)

# Filter to holds used in at least 20 routes for stable estimates
hold_stats_filtered = hold_stats[hold_stats["usage_count"] >= 20]

fig, axes = plt.subplots(1, 3, figsize=(18, 14))

# Mean grade heatmap on board
scatter1 = axes[0].scatter(
    hold_stats_filtered["x"],
    hold_stats_filtered["y"],
    c=hold_stats_filtered["mean_grade"],
    s=hold_stats_filtered["usage_count"] / hold_stats_filtered["usage_count"].max() * 150,
    cmap="RdYlGn_r",
    alpha=0.8,
    edgecolors="black",
    linewidths=0.3,
)
axes[0].set_xlabel("X position")
axes[0].set_ylabel("Y position")
axes[0].set_title("Mean Grade per Hold\n(size = usage count)")
plt.colorbar(scatter1, ax=axes[0], label="Mean grade")

# Grade variability (std) heatmap
scatter2 = axes[1].scatter(
    hold_stats_filtered["x"],
    hold_stats_filtered["y"],
    c=hold_stats_filtered["std_grade"],
    s=60,
    cmap="YlOrRd",
    alpha=0.8,
    edgecolors="black",
    linewidths=0.3,
)
axes[1].set_xlabel("X position")
axes[1].set_ylabel("Y position")
axes[1].set_title("Grade Variability (SD) per Hold")
plt.colorbar(scatter2, ax=axes[1], label="Grade SD")

# Mean angle heatmap
scatter3 = axes[2].scatter(
    hold_stats_filtered["x"],
    hold_stats_filtered["y"],
    c=hold_stats_filtered["mean_angle"],
    s=60,
    cmap="coolwarm",
    alpha=0.8,
    edgecolors="black",
    linewidths=0.3,
)
axes[2].set_xlabel("X position")
axes[2].set_ylabel("Y position")
axes[2].set_title("Mean Angle per Hold\n(red = used at steeper angles)")
plt.colorbar(scatter3, ax=axes[2], label="Mean angle (°)")

plt.tight_layout()
plt.show()

grade_min = hold_stats_filtered["mean_grade"].min()
grade_max = hold_stats_filtered["mean_grade"].max()
print(f"Holds with ≥20 uses: {len(hold_stats_filtered)}")
print(f"Mean grade range across holds: {grade_min:.1f} – {grade_max:.1f}")
print(
    f"This {grade_max - grade_min:.1f}-point spread confirms holds vary "
    "in difficulty — supporting the usability feature."
)

## 7. Save Processed Data to Parquet

In [ ]:
# Save processed climb data (without list columns — parquet can't store them directly)
# We'll save the frames string and re-parse as needed in feature engineering
df_save = df.drop(columns=["holds"]).copy()
df_save.to_parquet("../data/processed/climbs.parquet", index=False)
print(f"Saved climbs.parquet: {len(df_save):,} rows")

# Save placements
placements.to_parquet("../data/processed/placements.parquet", index=False)
print(f"Saved placements.parquet: {len(placements):,} rows")

# Save per-hold stats for later use in hold usability feature
hold_stats.to_parquet("../data/processed/hold_stats.parquet", index=False)
print(f"Saved hold_stats.parquet: {len(hold_stats):,} rows")